# Notebook 5: Topic Modelling (LDA + BERTopic)

## Purpose
Discover recurring **complaint themes** within KFC's negative Reddit comments using two approaches.

## Models
| Model | How it works | Strengths |
|---|---|---|
| **LDA** | Frequency-based (Bag-of-Words) | Broad, easy-to-interpret themes |
| **BERTopic** | Embedding-based (sentence-transformers + clustering) | Better with short/slangy Reddit text |

## Why both?
LDA gives broad themes. BERTopic captures nuance. Together they ensure we don't miss anything.

## Input
`kfc_negative_comments.csv` (from Notebook 4)

## Output
- `lda_topics.csv` — LDA topic keywords
- `bertopic_themes.csv` — BERTopic themes (add your interpretations!)
- `topic_comparison.csv` — side-by-side metrics
- `lda_coherence_plot.png`

## Step 0 — Import Libraries

In [ ]:
# !pip install pandas numpy gensim bertopic scikit-learn matplotlib
# !pip install sentence-transformers umap-learn hdbscan

import pandas as pd
import numpy as np
from gensim import corpora
from gensim.models import LdaModel, CoherenceModel
from bertopic import BERTopic
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings("ignore")

print("Libraries loaded.")

## Step 1 — Load Negative Comments

In [ ]:
INPUT_FILE = "kfc_negative_comments.csv"

df = pd.read_csv(INPUT_FILE)
print(f"Loaded {len(df)} negative comments")

documents = df["Tokenised_Comment"].tolist()
tokenised_docs = [doc.split() for doc in documents]

---
# Part A: LDA (Latent Dirichlet Allocation)

LDA is a traditional topic model. It treats each document as a mixture of topics, and each topic as a mixture of words. It works on **word frequency** (Bag-of-Words).

## Step 2 — Build BoW Corpus for LDA

In [ ]:
# Create a dictionary (maps each unique word to an ID)
dictionary = corpora.Dictionary(tokenised_docs)
dictionary.filter_extremes(no_below=5, no_above=0.5)

# Convert each document to a list of (word_id, count) tuples
corpus = [dictionary.doc2bow(doc) for doc in tokenised_docs]

print(f"Dictionary: {len(dictionary)} unique tokens")
print(f"Corpus: {len(corpus)} documents")

## Step 3 — Find Optimal Number of Topics

We train LDA with 2–10 topics and pick the one with the highest **coherence score**.  
Coherence measures how semantically similar the top words within each topic are (higher = better).

In [ ]:
print("Testing LDA with 2 to 10 topics...")
topic_range = range(2, 11)
coherence_scores = []

for n in topic_range:
    model = LdaModel(corpus=corpus, id2word=dictionary, num_topics=n,
                     random_state=42, passes=15, alpha="auto", eta="auto")
    cm = CoherenceModel(model=model, texts=tokenised_docs,
                        dictionary=dictionary, coherence="c_v")
    score = cm.get_coherence()
    coherence_scores.append(score)
    print(f"  Topics={n:2d}  Coherence={score:.4f}")

best_idx = np.argmax(coherence_scores)
best_n = list(topic_range)[best_idx]
best_coh = coherence_scores[best_idx]
print(f"\nBest: {best_n} topics (coherence = {best_coh:.4f})")

## Step 4 — Coherence Plot

In [ ]:
plt.figure(figsize=(10, 5))
plt.plot(list(topic_range), coherence_scores, "o-", linewidth=2, markersize=8)
plt.axvline(x=best_n, color="red", linestyle="--", label=f"Best: {best_n} topics")
plt.xlabel("Number of Topics"); plt.ylabel("Coherence Score (c_v)")
plt.title("LDA Coherence Scores — KFC Negative Comments")
plt.legend(); plt.grid(True, alpha=0.3); plt.tight_layout()
plt.savefig("lda_coherence_plot.png", dpi=150)
plt.show()

## Step 5 — Train Final LDA Model & Extract Topics

In [ ]:
print(f"Training final LDA with {best_n} topics...")
final_lda = LdaModel(corpus=corpus, id2word=dictionary, num_topics=best_n,
                     random_state=42, passes=20, alpha="auto", eta="auto")

lda_data = []
print(f"\nLDA Topics (top 10 keywords):")
for idx in range(best_n):
    keywords = [w for w, _ in final_lda.show_topic(idx, topn=10)]
    print(f"  Topic {idx}: {', '.join(keywords)}")
    lda_data.append({"Topic": idx, "Top Keywords": ", ".join(keywords)})

pd.DataFrame(lda_data).to_csv("lda_topics.csv", index=False)

# Diversity
all_w = [w for i in range(best_n) for w, _ in final_lda.show_topic(i, topn=10)]
lda_div = len(set(all_w)) / len(all_w)
print(f"\nLDA Diversity: {lda_div:.3f}")

---
# Part B: BERTopic

BERTopic uses **contextual sentence embeddings** (not word counts) to cluster similar documents, then extracts keywords per cluster using class-based TF-IDF.

Pipeline: Embeddings → UMAP dimensionality reduction → HDBSCAN clustering → c-TF-IDF keywords

## Step 6 — Train BERTopic

In [ ]:
print("Training BERTopic (this may take a few minutes)...")

# BERTopic uses the raw comment text (not tokenised) for embedding
raw_docs = df["Comment"].tolist()

topic_model = BERTopic(
    language="english",
    min_topic_size=10,
    nr_topics="auto",
    verbose=True,
)

topics, probs = topic_model.fit_transform(raw_docs)
print("\nBERTopic training complete.")

## Step 7 — Extract BERTopic Results

In [ ]:
topic_info = topic_model.get_topic_info()
n_topics = len(topic_info) - 1  # exclude outlier topic -1
outliers = topic_info[topic_info["Topic"] == -1]["Count"].values[0]

print(f"Topics found: {n_topics} (excluding outlier topic -1)")
print(f"Outlier comments: {outliers}")

bert_data = []
print(f"\nBERTopic Topics:")
for _, row in topic_info[topic_info["Topic"] != -1].iterrows():
    tid = row["Topic"]
    kw = [w for w, _ in topic_model.get_topic(tid)[:10]]
    print(f"  Topic {tid}: {', '.join(kw)}")
    bert_data.append({
        "Topic": tid, "Count": row["Count"],
        "Top Keywords": ", ".join(kw), "Interpreted Theme": ""
    })

pd.DataFrame(bert_data).to_csv("bertopic_themes.csv", index=False)
print("\nSaved: bertopic_themes.csv (fill in Interpreted Theme column!)") 

## Step 8 — Calculate BERTopic Coherence & Diversity

In [ ]:
all_bw = []
tw_lists = []
for _, row in topic_info[topic_info["Topic"] != -1].iterrows():
    kw = [w for w, _ in topic_model.get_topic(row["Topic"])[:10]]
    all_bw.extend(kw); tw_lists.append(kw)

bert_div = len(set(all_bw)) / len(all_bw) if all_bw else 0

try:
    bcm = CoherenceModel(topics=tw_lists, texts=tokenised_docs,
                         dictionary=dictionary, coherence="c_v")
    bert_coh = bcm.get_coherence()
    print(f"BERTopic Coherence: {bert_coh:.4f}")
except Exception as e:
    bert_coh = None
    print(f"Could not compute coherence: {e}")

print(f"BERTopic Diversity: {bert_div:.3f}")

---
# Part C: Model Comparison

In [ ]:
comparison = pd.DataFrame({
    "Metric": ["Coherence Score", "Topic Diversity", "Number of Topics"],
    "LDA": [f"{best_coh:.3f}", f"{lda_div:.3f}", str(best_n)],
    "BERTopic": [f"{bert_coh:.3f}" if bert_coh else "N/A", f"{bert_div:.3f}", str(n_topics)],
})

print("MODEL COMPARISON")
print("=" * 50)
print(comparison.to_string(index=False))
comparison.to_csv("topic_comparison.csv", index=False)
print("\nSaved: topic_comparison.csv")

## Next Step

Open `bertopic_themes.csv` and fill in the **Interpreted Theme** column by reading each topic's keywords and representative comments. This human validation step is essential for generating business recommendations.

Example interpretations (from the McDonald's study):
- "app, deal, sold, available, use" → *Mobile app issues, deal availability frustrations*
- "machine, cream, ice, broken" → *Equipment malfunctions (e.g., ice cream machines)*